## Step 1: Setting up the environment

I am using google colab kernel for this notebook

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np

# Confirm versions 
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# We will use CPU throughout — no GPU needed to understand the architecture
device = torch.device('cpu')
print("Using device:", device)

PyTorch version: 2.11.0+cpu
CUDA available: False
Using device: cpu


In [2]:
d_model     = 512
num_heads   = 8
d_ff        = 2048
num_layers  = 6
dropout     = 0.1
max_seq_len = 100
vocab_size  = 1000

## Step 2 : Input Embedding Class

**InputEmbedding: converts integer token IDs into 512-dimensional dense vectors and scales them by sqrt(d_model) so they are on the same magnitude as positional encodings.**

In [3]:
class InputEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)   # nn.Embedding is a lookup table 
        self.d_model = d_model             # The self. prefix means it belongs to this object and will be accessible in the forward method.

    def forward(self, x):   # Every nn.Module must have a forward method .This defines what happens to the data when it passes through this component. 
        # x shape coming in : (batch_size, seq_len)  — integer token IDs e.g (2,5) : 2 sentences, each with 5 tokens
        # x shape going out : (batch_size, seq_len, d_model) e.g (2,5,512) : each token is now represented by a 512-dimensional vector
        return self.embedding(x) * math.sqrt(self.d_model)  # the math.sqrt(d_model) scaling is multiplied to the embeddings to scale them up. This is followed from the original transformer paper.

In [4]:
# Test InputEmbedding
embed = InputEmbedding(vocab_size, d_model)

# Fake input: batch of 2 sentences, each 5 tokens long
# Each number is a token ID (integer between 0 and vocab_size-1)
dummy_tokens = torch.tensor([
    [4, 27, 103, 56, 8],   # sentence 1
    [9, 41, 7,  200, 3],   # sentence 2
])

print("Input shape  :", dummy_tokens.shape)   # (2, 5)

output = embed(dummy_tokens)
print("Output shape :", output.shape)          # (2, 5, 512)
print("Sample vector (first token, first sentence):", output[0][0][:5])

Input shape  : torch.Size([2, 5])
Output shape : torch.Size([2, 5, 512])
Sample vector (first token, first sentence): tensor([ -0.1190,  19.5242, -57.3267,  13.3650,  22.8943],
       grad_fn=<SliceBackward0>)


## Step 3 : Positional Encoding

**PositionalEncoding: adds fixed sine and cosine patterns to the embeddings so the model knows the position of each token in the sequence. Shape never changes — it only adds information to the values.**

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_seq_len, d_model)

        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]       # x.size(1) returns the actual sequence length of the current input. In our test it is 5. So `self.pe[:, :5, :]` slices the first 5 rows from our 100-row PE matrix. Shape: `(1, 5, 512)`.

        return self.dropout(x)

In [6]:
## Positional Encoding test
pos_enc = PositionalEncoding(d_model, max_seq_len, dropout)

pe_output = pos_enc(output)

print("Input shape :", output.shape)
print("Output shape:", pe_output.shape)
print()
print("Before PE:", output[0][0][:5])
print("After PE :", pe_output[0][0][:5])

Input shape : torch.Size([2, 5, 512])
Output shape: torch.Size([2, 5, 512])

Before PE: tensor([ -0.1190,  19.5242, -57.3267,  13.3650,  22.8943],
       grad_fn=<SliceBackward0>)
After PE : tensor([ -0.1323,  22.8046, -63.6964,   0.0000,  25.4381],
       grad_fn=<SliceBackward0>)


## Step 4 : Scaled Dot-Product Attention

In [7]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    # Q shape: (batch_size, num_heads, seq_len, d_k)
    # K shape: (batch_size, num_heads, seq_len, d_k)
    # V shape: (batch_size, num_heads, seq_len, d_k)

    d_k = Q.size(-1)

    # Step 1: compute raw attention scores
    # Q @ K.transpose(-2,-1) shape: (batch_size, num_heads, seq_len, seq_len)   ## K.transpose(-2, -1) swaps the last two dimensions of K. K shape was (2, 8, 5, 64). After transpose it becomes (2, 8, 64, 5)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    # Step 2: apply mask if provided
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    # Step 3: softmax over last dimension
    attention_weights = F.softmax(scores, dim=-1)

    # Step 4: multiply by V
    # output shape: (batch_size, num_heads, seq_len, d_k)
    output = torch.matmul(attention_weights, V)

    return output, attention_weights

In [8]:
## Scaled Dot-Product Attention test
batch_size = 2
num_heads  = 8
seq_len    = 5
d_k        = d_model // num_heads   # 512 // 8 = 64

# Fake Q, K, V matrices
Q = torch.randn(batch_size, num_heads, seq_len, d_k)
K = torch.randn(batch_size, num_heads, seq_len, d_k)
V = torch.randn(batch_size, num_heads, seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)

print("Q shape             :", Q.shape)
print("K shape             :", K.shape)
print("V shape             :", V.shape)
print("Output shape        :", output.shape)
print("Attention weights   :", weights.shape)
print()
print("Weights for first head, first sentence, first token:")
print(weights[0][0][0])
print()
print("Do weights sum to 1?", weights[0][0][0].sum().item())

Q shape             : torch.Size([2, 8, 5, 64])
K shape             : torch.Size([2, 8, 5, 64])
V shape             : torch.Size([2, 8, 5, 64])
Output shape        : torch.Size([2, 8, 5, 64])
Attention weights   : torch.Size([2, 8, 5, 5])

Weights for first head, first sentence, first token:
tensor([0.4429, 0.3498, 0.0631, 0.0893, 0.0549])

Do weights sum to 1? 1.0000001192092896


## Step 5 : Multi-Head Attention

In [9]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch_size = x.size(0)
        x = x.view(batch_size, -1, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))  # 
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)

        batch_size = attn_output.size(0)
        attn_output = attn_output.transpose(1, 2)
        attn_output = attn_output.contiguous().view(batch_size, -1, self.d_model)

        return self.W_o(attn_output)

In [33]:
## Multi-Head Attention test
mha = MultiHeadAttention(d_model, num_heads)

x = torch.randn(2, 5, 512)

output = mha(x, x, x)
print("Input shape :", x.shape)
print("Output shape:", output.shape)

Input shape : torch.Size([2, 5, 512])
Output shape: torch.Size([2, 5, 512])
